In [75]:
#ProjectAlex - A simple Python script to generate clear data
#Kiarash Geraili

In [82]:
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
from matplotlib.dates import date2num
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import os
import glob
import pandas as pd

Rana Bluesky Data Cleaning

In [ ]:
df_rana_unclean = pd.read_csv('bluesky_rana_tagged_data_raw/rana_bluesky_tagged_data.csv', dtype=str)
# Clean the DataFrame
df_rana_unclean = df_rana_unclean[df_rana_unclean['Educator'].notna()].copy()
df_rana_unclean = df_rana_unclean[df_rana_unclean['text'].notna()].copy()
df_rana_unclean = df_rana_unclean.dropna(axis=1, how='all')
df_rana_unclean

,Unnamed: 0,Unnamed: 1,Unnamed: 2,text,Unnamed: 4,ID,Unnamed: 6,Educator,Student,Expert/keynote/researchers,adv/marketing,public/can't figure out,Unnamed: 12,Unnamed: 13,Category,Specific
0,at://did:plc:rygblwb6y4ettstfzqcyw3mp/app.bsky...,Matt Reece,did:plc:rygblwb6y4ettstfzqcyw3mp,The latest email from a Dean encouraging us to...,NaN,https://bsky.app/profile/mreece.bsky.social/po...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,Educator,HE
1,at://did:plc:tbqqvyv6pjjww44glrmycaxl/app.bsky...,Carl T. Bergstrom,did:plc:tbqqvyv6pjjww44glrmycaxl,They looked at the performance of 7 different ...,at://did:plc:tbqqvyv6pjjww44glrmycaxl/app.bsky...,https://bsky.app/profile/carlbergstrom.com/pos...,NaN,1,NaN,NaN,NaN,0,NaN,NaN,Educator,Prof
2,at://did:plc:x3vqyrdzkttyiyfsbclyynmi/app.bsky...,Glenda Sims,did:plc:x3vqyrdzkttyiyfsbclyynmi,"""One thing that makes me optimistic...is that ...",NaN,https://bsky.app/profile/gwitch.bsky.social/po...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,NaN
3,at://did:plc:nazzzoawtvkr62s25ggrrjvv/app.bsky...,tea,did:plc:nazzzoawtvkr62s25ggrrjvv,I don't put much value in my art degree (it on...,NaN,https://bsky.app/profile/steelpeach.bsky.socia...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,artist
4,at://did:plc:slyb6dtlemv6ucoymlcufuan/app.bsky...,Rod Trent,did:plc:slyb6dtlemv6ucoymlcufuan,AI Level-Up for Teachers https://rodtrent.com/ock,NaN,https://bsky.app/profile/rodtrent.bsky.social/...,NaN,0,NaN,NaN,1,NaN,NaN,NaN,Knowledgable,Computer scientist
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31233,at://did:plc:m5uvdbeipojafbybfdvczbna/app.bsky...,m. (taylor’s version),did:plc:m5uvdbeipojafbybfdvczbna,"to morta de doente ne, ai pensei “hmmm entao n...",NaN,https://bsky.app/profile/yomag.bsky.social/pos...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,NaN
31234,at://did:plc:frx7obbsqel7bpm6544f4roj/app.bsky...,Liz Ahl,did:plc:frx7obbsqel7bpm6544f4roj,Reminds me of the online/MOOC rush. To complet...,at://did:plc:hakzua5btcgcfjye2yujlhos/app.bsky...,https://bsky.app/profile/surlyacres.bsky.socia...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,Educator,Teacher
31235,at://did:plc:7nnjvnyzx6bxparchrfqqh46/app.bsky...,Katherine Stiles,did:plc:7nnjvnyzx6bxparchrfqqh46,In an article published online on Oct. 10 in t...,NaN,https://bsky.app/profile/katherinestiles.org/p...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,Educator,HE
31236,at://did:plc:kehtdprodjwwr7tzq3bhdr4g/app.bsky...,Starlit_Stella,did:plc:kehtdprodjwwr7tzq3bhdr4g,Trump voters apparently. I give up. Seriously...,at://did:plc:y223crzq4mk4gue4q5eniq5c/app.bsky...,https://bsky.app/profile/starlilstella.bsky.so...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,NaN


In [84]:
non_null_counts = df_rana_unclean.notna().sum()
print("Non-null counts per column:\n", non_null_counts)

Non-null counts per column:
 Unnamed: 0                    2009
Unnamed: 1                    1970
Unnamed: 2                    2009
text                          2009
Unnamed: 4                     740
ID                            2008
Unnamed: 6                       1
Educator                      2009
Student                         15
Expert/keynote/researchers      66
adv/marketing                  204
public/can't figure out       1200
Unnamed: 12                     10
Unnamed: 13                      8
Category                      2002
Specific                       730
dtype: int64


In [85]:
cols = ['Unnamed: 6', 'Unnamed: 12', 'Unnamed: 13']
for col in cols:
    if col in df_rana_unclean.columns:
        uniques = df_rana_unclean[col].dropna().unique()
        print(f"Unique values in column '{col}':")
        print(uniques, "\n")
    else:
        print(f"Column '{col}' not found in DataFrame.\n")
        
# 4) Remove rows missing essential information
df_rana_unclean.dropna(subset=["text", "Educator"], inplace=True)

# 6) Deduplicate on the text field
before = len(df_rana_unclean)
df_rana_unclean.drop_duplicates(subset=["text"], inplace=True)
after = len(df_rana_unclean)

print(f"Dropped {before-after} duplicate text rows")
df_rana_unclean

Unique values in column 'Unnamed: 6':
['https://bsky.app/profile/dsi-uchicago.bsky.social/post/3l6inpafljf23'] 

Unique values in column 'Unnamed: 12':
['1'] 

Unique values in column 'Unnamed: 13':
['NK' 'Educator'] 

Dropped 7 duplicate text rows


,Unnamed: 0,Unnamed: 1,Unnamed: 2,text,Unnamed: 4,ID,Unnamed: 6,Educator,Student,Expert/keynote/researchers,adv/marketing,public/can't figure out,Unnamed: 12,Unnamed: 13,Category,Specific
0,at://did:plc:rygblwb6y4ettstfzqcyw3mp/app.bsky...,Matt Reece,did:plc:rygblwb6y4ettstfzqcyw3mp,The latest email from a Dean encouraging us to...,NaN,https://bsky.app/profile/mreece.bsky.social/po...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,Educator,HE
1,at://did:plc:tbqqvyv6pjjww44glrmycaxl/app.bsky...,Carl T. Bergstrom,did:plc:tbqqvyv6pjjww44glrmycaxl,They looked at the performance of 7 different ...,at://did:plc:tbqqvyv6pjjww44glrmycaxl/app.bsky...,https://bsky.app/profile/carlbergstrom.com/pos...,NaN,1,NaN,NaN,NaN,0,NaN,NaN,Educator,Prof
2,at://did:plc:x3vqyrdzkttyiyfsbclyynmi/app.bsky...,Glenda Sims,did:plc:x3vqyrdzkttyiyfsbclyynmi,"""One thing that makes me optimistic...is that ...",NaN,https://bsky.app/profile/gwitch.bsky.social/po...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,NaN
3,at://did:plc:nazzzoawtvkr62s25ggrrjvv/app.bsky...,tea,did:plc:nazzzoawtvkr62s25ggrrjvv,I don't put much value in my art degree (it on...,NaN,https://bsky.app/profile/steelpeach.bsky.socia...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,artist
4,at://did:plc:slyb6dtlemv6ucoymlcufuan/app.bsky...,Rod Trent,did:plc:slyb6dtlemv6ucoymlcufuan,AI Level-Up for Teachers https://rodtrent.com/ock,NaN,https://bsky.app/profile/rodtrent.bsky.social/...,NaN,0,NaN,NaN,1,NaN,NaN,NaN,Knowledgable,Computer scientist
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31233,at://did:plc:m5uvdbeipojafbybfdvczbna/app.bsky...,m. (taylor’s version),did:plc:m5uvdbeipojafbybfdvczbna,"to morta de doente ne, ai pensei “hmmm entao n...",NaN,https://bsky.app/profile/yomag.bsky.social/pos...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,NaN
31234,at://did:plc:frx7obbsqel7bpm6544f4roj/app.bsky...,Liz Ahl,did:plc:frx7obbsqel7bpm6544f4roj,Reminds me of the online/MOOC rush. To complet...,at://did:plc:hakzua5btcgcfjye2yujlhos/app.bsky...,https://bsky.app/profile/surlyacres.bsky.socia...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,Educator,Teacher
31235,at://did:plc:7nnjvnyzx6bxparchrfqqh46/app.bsky...,Katherine Stiles,did:plc:7nnjvnyzx6bxparchrfqqh46,In an article published online on Oct. 10 in t...,NaN,https://bsky.app/profile/katherinestiles.org/p...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,Educator,HE
31236,at://did:plc:kehtdprodjwwr7tzq3bhdr4g/app.bsky...,Starlit_Stella,did:plc:kehtdprodjwwr7tzq3bhdr4g,Trump voters apparently. I give up. Seriously...,at://did:plc:y223crzq4mk4gue4q5eniq5c/app.bsky...,https://bsky.app/profile/starlilstella.bsky.so...,NaN,0,NaN,NaN,NaN,1,NaN,NaN,NK,NaN


In [ ]:
# 1) Rename the specified columns
df_rana_unclean = df_rana_unclean.rename(columns={
    "Unnamed: 0": "uri",
    "Unnamed: 1": "Author",
    "Unnamed: 2": "Author_did",
    "Unnamed: 4": "Reply"
})

# 2) Keep only the desired columns
keep_cols = ["uri", "Author", "ID", "Author_did", "Reply", "text", "Educator"]
df_rana_unclean = df_rana_unclean[keep_cols]

# 3) (Optional) Inspect the result
df_rana_unclean
# Save the cleaned DataFrame to a CSV file
df_rana_unclean.to_csv("rana_tagged_data_clean/bluesky_rana_tagged_clean.csv", index = False)



In [87]:
df_rana_unclean

,uri,Author,ID,Author_did,Reply,text,Educator
0,at://did:plc:rygblwb6y4ettstfzqcyw3mp/app.bsky...,Matt Reece,https://bsky.app/profile/mreece.bsky.social/po...,did:plc:rygblwb6y4ettstfzqcyw3mp,NaN,The latest email from a Dean encouraging us to...,1
1,at://did:plc:tbqqvyv6pjjww44glrmycaxl/app.bsky...,Carl T. Bergstrom,https://bsky.app/profile/carlbergstrom.com/pos...,did:plc:tbqqvyv6pjjww44glrmycaxl,at://did:plc:tbqqvyv6pjjww44glrmycaxl/app.bsky...,They looked at the performance of 7 different ...,1
2,at://did:plc:x3vqyrdzkttyiyfsbclyynmi/app.bsky...,Glenda Sims,https://bsky.app/profile/gwitch.bsky.social/po...,did:plc:x3vqyrdzkttyiyfsbclyynmi,NaN,"""One thing that makes me optimistic...is that ...",0
3,at://did:plc:nazzzoawtvkr62s25ggrrjvv/app.bsky...,tea,https://bsky.app/profile/steelpeach.bsky.socia...,did:plc:nazzzoawtvkr62s25ggrrjvv,NaN,I don't put much value in my art degree (it on...,0
4,at://did:plc:slyb6dtlemv6ucoymlcufuan/app.bsky...,Rod Trent,https://bsky.app/profile/rodtrent.bsky.social/...,did:plc:slyb6dtlemv6ucoymlcufuan,NaN,AI Level-Up for Teachers https://rodtrent.com/ock,0
...,...,...,...,...,...,...,...
31233,at://did:plc:m5uvdbeipojafbybfdvczbna/app.bsky...,m. (taylor’s version),https://bsky.app/profile/yomag.bsky.social/pos...,did:plc:m5uvdbeipojafbybfdvczbna,NaN,"to morta de doente ne, ai pensei “hmmm entao n...",0
31234,at://did:plc:frx7obbsqel7bpm6544f4roj/app.bsky...,Liz Ahl,https://bsky.app/profile/surlyacres.bsky.socia...,did:plc:frx7obbsqel7bpm6544f4roj,at://did:plc:hakzua5btcgcfjye2yujlhos/app.bsky...,Reminds me of the online/MOOC rush. To complet...,1
31235,at://did:plc:7nnjvnyzx6bxparchrfqqh46/app.bsky...,Katherine Stiles,https://bsky.app/profile/katherinestiles.org/p...,did:plc:7nnjvnyzx6bxparchrfqqh46,NaN,In an article published online on Oct. 10 in t...,1
31236,at://did:plc:kehtdprodjwwr7tzq3bhdr4g/app.bsky...,Starlit_Stella,https://bsky.app/profile/starlilstella.bsky.so...,did:plc:kehtdprodjwwr7tzq3bhdr4g,at://did:plc:y223crzq4mk4gue4q5eniq5c/app.bsky...,Trump voters apparently. I give up. Seriously...,0


**Rana Tweeter Data Preprocessing**

In [ ]:
import os
import pandas as pd

# 1. Define the folder containing your raw files
folder_path = "tweet_rana_tagged_data_raw"

# 2. Collect all Excel and CSV filenames in the folder
raw_files = [
    f for f in os.listdir(folder_path)
    if f.lower().endswith((".xlsx", ".xls", ".csv"))
]

# 3. Read each file appropriately, filter for rows with a non-NaN 'id', and accumulate
dfs = []
for filename in raw_files:
    file_path = os.path.join(folder_path, filename)
    ext = os.path.splitext(filename)[1].lower()
    
    try:
        if ext in [".xlsx", ".xls"]:
            df = pd.read_excel(file_path)
        elif ext == ".csv":
            df = pd.read_csv(file_path)
        else:
            continue  # skip any other extensions
    except Exception as e:
        print(f"Warning: could not read {filename}: {e}")
        continue
    
    if "id" not in df.columns:
        print(f"Warning: 'id' column not found in {filename}, skipping.")
        continue
    
    # Keep only rows where 'id' is not null
    df = df[df["id"].notna()]
    dfs.append(df)

# 4. Concatenate all DataFrames into one
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"Combined {len(dfs)} files into a DataFrame of shape {combined_df.shape}")
    
    # 5. (Optional) Save the combined DataFrame to a new file
    combined_df.to_csv("combined_rana_tagged_data.csv", index=False)
    print("Saved combined DataFrame to 'combined_rana_tagged_data.csv'")
else:
    print("No valid dataframes to concatenate.")


Combined 5 files into a DataFrame of shape (2205, 26)
Saved combined DataFrame to 'combined_rana_tagged_data.csv'


In [67]:
combined_df

,Unnamed: 0,id,text,Unnamed: 3,Educator,Student,Expert/keynote/researchers,ad/websites/public/can't figure out,index,Business,...,Unnamed: 10,Teacher-ChatGPT,Business-ChatGPT,created_at,Unnamed: 4,business,public,n,resercher/expert,public/adv/can not figure out
0,0.0,1.820000e+18,Book me for your next event. I am available fo...,NaN,0,0.0,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,1.820000e+18,"#ChatGPT for #education: Analysis, evaluation ...",NaN,0,0.0,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2.0,1.820000e+18,"AI Course Creator is NOW LIVE, a revolutionary...",NaN,1,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3.0,1.820000e+18,3) What was your experience like testing DALL-...,NaN,1,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4.0,1.820000e+18,How are you leveraging @getpostman to test and...,NaN,0,0.0,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2200,495.0,1.819285e+18,GitHub Models is a new feature that allows use...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
2201,496.0,1.819285e+18,Check out my new Oracle Cloud Infrastructure 2...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
2202,497.0,1.819282e+18,Have you used videos to help train a new emplo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
2203,498.0,1.819282e+18,Have you used videos to help train a new emplo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN


In [ ]:
# manually adding numbers of rana tagged data to be sure it is consistent with the original data
690+227+98+500+690

2205

In [90]:
# 2) Keep only the desired columns
# consider that: some of these files do not have date, so in the final result I'll drop date since we do not have access to it
keep_cols = ["id", "text", "Educator"]
combined_df = combined_df[keep_cols]

# Replace NaN values in the 'Educator' column with 0
combined_df['Educator'] = combined_df['Educator'].fillna(0)

# If you want to ensure the column is integer type afterward:
combined_df['Educator'] = combined_df['Educator'].astype(int)


# 3) (Optional) Inspect the result
combined_df

,id,text,Educator
0,1.820000e+18,Book me for your next event. I am available fo...,0
1,1.820000e+18,"#ChatGPT for #education: Analysis, evaluation ...",0
2,1.820000e+18,"AI Course Creator is NOW LIVE, a revolutionary...",1
3,1.820000e+18,3) What was your experience like testing DALL-...,1
4,1.820000e+18,How are you leveraging @getpostman to test and...,0
...,...,...,...
2200,1.819285e+18,GitHub Models is a new feature that allows use...,0
2201,1.819285e+18,Check out my new Oracle Cloud Infrastructure 2...,0
2202,1.819282e+18,Have you used videos to help train a new emplo...,0
2203,1.819282e+18,Have you used videos to help train a new emplo...,0


In [ ]:
# Save the cleaned DataFrame to a CSV file
combined_df.to_csv("rana_tagged_data_clean/tweeter_rana_tagged_clean.csv", index = False)